In [1]:
import sys
!{sys.executable} -m pip install robin_stocks


In [2]:
import sys
from pathlib import Path

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebook" else cwd
PROJECT_ROOT = project_root
BASE_DIR = str(PROJECT_ROOT)
notebook_dir = project_root / "notebook"

for path in (project_root, notebook_dir):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))


In [3]:
import sys
!{sys.executable} -m pip install openpyxl


In [4]:
from robinhood_auth_login import login
import os
import time

print("Starting Robinhood login...")

rh_user = os.getenv("ROBINHOOD_USERNAME")
rh_pass = os.getenv("ROBINHOOD_PASSWORD")
if not rh_user or not rh_pass:
    raise ValueError("Missing ROBINHOOD_USERNAME or ROBINHOOD_PASSWORD in environment.")

t0 = time.time()
login_response = login(
    username=rh_user,
    password=rh_pass,
    by_sms=True,
    store_session=False
)
print(f"Login returned after {time.time()-t0:.1f}s")

with open("robinhood_token.txt", "w") as f:
    f.write(login_response["access_token"])

print("✅ Token saved to robinhood_token.txt")

Starting Robinhood login...


Verification workflow required. Please check your Robinhood app for instructions.


Waiting for challenge to be validated
5.116610288619995


Waiting for challenge to be validated
10.2807936668396


Waiting for challenge to be validated
15.393234968185425


...

Waiting for challenge to be validated
107.51084017753601


Waiting for challenge to be validated
112.63378715515137


Waiting for challenge to be validated
117.73600196838379


Waiting for challenge to be validated
122.85842609405518


Exception: Login confirmation timed out. Please try again.

In [ ]:
import os, sys
print("cwd:", os.getcwd())
print("files here:", [f for f in os.listdir() if f.startswith("robinhood")])
print("sys.path[0]:", sys.path[0])


In [ ]:
import requests
import pandas as pd
from datetime import datetime
import os  

# Load token from file
def load_token_from_file(file_path="robinhood_token.txt"):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            return f.read().strip()
    return None

# Fetch and save stock data
def fetch_and_save_stock_data(token, output_file="my_stock_data.xlsx"):
    url = "https://api.robinhood.com/positions/"
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Error fetching stock data:", response.text)
        return

    positions = response.json().get("results", [])
    stock_data = []

    # Get the current timestamp
    fetch_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    def get_quote_price(symbol):
        quote_url = f"https://api.robinhood.com/quotes/{symbol}/"
        quote_response = requests.get(quote_url, headers=headers)
        if quote_response.status_code != 200:
            print(f"Skipping {symbol}: quote request failed ({quote_response.status_code})")
            return None

        quote_data = quote_response.json()
        for key in ("last_trade_price", "last_extended_hours_trade_price", "previous_close"):
            value = quote_data.get(key)
            if value not in (None, ""):
                return float(value)

        print(f"Skipping {symbol}: quote response did not include a price. Keys: {sorted(quote_data.keys())}")
        return None

    for position in positions:
        if float(position["quantity"]) > 0:  # Only fetch open positions
            instrument_url = position["instrument"]
            instrument_data = requests.get(instrument_url, headers=headers).json()

            symbol = instrument_data["symbol"]
            quantity = float(position["quantity"])
            average_buy_price = float(position["average_buy_price"])
            current_price = get_quote_price(symbol)
            if current_price is None:
                continue

            market_value = current_price * quantity
            profit_loss = (current_price - average_buy_price) * quantity

            stock_data.append({
                "Timestamp": fetch_time,  # Add the timestamp
                "Symbol": symbol,
                "Quantity": quantity,
                "Average Buy Price": average_buy_price,
                "Current Price": current_price,
                "Market Value": market_value,
                "Profit/Loss": profit_loss
            })

    df = pd.DataFrame(stock_data)
    df.to_excel(output_file, index=False)
    print(f"Stock data saved to {output_file}")

# Use token to fetch data
token = load_token_from_file()
if token:
    fetch_and_save_stock_data(token)
else:
    print("No token found. Please log in first.")


In [ ]:
import requests
import pandas as pd
import time, random
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import sqlite3
import os
from collections import Counter

# === CONFIG ===
BASE_DIR = str(PROJECT_ROOT)
os.makedirs(BASE_DIR, exist_ok=True)

RATE_LIMIT_DELAY = 1
MAX_WORKERS = 4
MIN_VOLUME = 100000
MIN_DOLLAR_VOLUME = 1_000_000
MAX_STDDEV = 0.05
MIN_PRICE = 3.0
MOMENTUM_LOOKBACK_DAYS = 5
SAVE_INTERVAL = 50
RETRY_FAILED_ONLY = False

# Incremental/refresh behavior
REFRESH_DAYS = 3  # skip re-processing if checked within this many days

# Optional trade-frequency filter (proxy)
USE_TRADE_FREQ_FILTER = False
ASSUMED_AVG_TRADE_SIZE = 300   # shares/trade (proxy)
SESSION_MINUTES = 390          # US regular session
MIN_TRADES_PER_MIN = 0.5       # e.g., at least a trade every 2 minutes

# === FILE PATHS ===
TOKEN_FILE = os.path.join(BASE_DIR, "robinhood_token.txt")
CHECKPOINT_FILE = os.path.join(BASE_DIR, "checkpoint_filtered.csv")
REJECTED_FILE = os.path.join(BASE_DIR, "checkpoint_rejected.csv")
DB_FILE = os.path.join(BASE_DIR, "filtered_tickers.db")

# === TOKEN ===
def load_token_from_file(file_path=TOKEN_FILE):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            return f.read().strip()
    return None

token = load_token_from_file()
HEADERS = {"Authorization": f"Bearer {token}"} if token else {}

# shared requests session
SESSION = requests.Session()

# === LOGGING ===
def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

# === DB INIT & MIGRATION (idempotent) ===
def init_db(db_path=DB_FILE):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # Create tables if missing (minimal schema; we enforce uniqueness via index)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS FilteredTickers (
            Ticker TEXT,
            Name TEXT,
            Price REAL,
            LastChecked TEXT
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS RejectedTickers (
            Ticker TEXT,
            Reason TEXT,
            LastChecked TEXT
        )
    """)

    # Ensure LastChecked exists (for legacy DBs)
    cur.execute("PRAGMA table_info(FilteredTickers)")
    fcols = {r[1] for r in cur.fetchall()}
    if "LastChecked" not in fcols:
        cur.execute("ALTER TABLE FilteredTickers ADD COLUMN LastChecked TEXT")

    cur.execute("PRAGMA table_info(RejectedTickers)")
    rcols = {r[1] for r in cur.fetchall()}
    if "LastChecked" not in rcols:
        cur.execute("ALTER TABLE RejectedTickers ADD COLUMN LastChecked TEXT")

    # De-duplicate legacy rows (keep earliest rowid per ticker to be safe)
    for tbl in ("FilteredTickers","RejectedTickers"):
        cur.execute(f"""
        DELETE FROM {tbl}
        WHERE rowid NOT IN (
            SELECT MIN(rowid) FROM {tbl} GROUP BY Ticker
        )
        """)

    # Ensure UNIQUE index on Ticker so ON CONFLICT(Ticker) works
    cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS uq_filtered_ticker ON FilteredTickers(Ticker)")
    cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS uq_rejected_ticker ON RejectedTickers(Ticker)")

    # Helpful indexes
    cur.execute("CREATE INDEX IF NOT EXISTS idx_filtered_last ON FilteredTickers(LastChecked)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_rejected_last ON RejectedTickers(LastChecked)")

    conn.commit()
    conn.close()

def upsert_filtered(row, db_path=DB_FILE):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO FilteredTickers(Ticker, Name, Price, LastChecked)
        VALUES(?,?,?,?)
        ON CONFLICT(Ticker) DO UPDATE SET
            Name=excluded.Name,
            Price=excluded.Price,
            LastChecked=excluded.LastChecked
    """, (row["Ticker"], row.get("Name",""), row.get("Price"), row.get("LastChecked")))
    conn.commit()
    conn.close()

def upsert_rejected(ticker, reason, db_path=DB_FILE):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO RejectedTickers(Ticker, Reason, LastChecked)
        VALUES(?,?,?)
        ON CONFLICT(Ticker) DO UPDATE SET
            Reason=excluded.Reason,
            LastChecked=excluded.LastChecked
    """, (ticker, reason, datetime.utcnow().isoformat()))
    conn.commit()
    conn.close()

def load_recently_checked(db_path=DB_FILE, refresh_days=REFRESH_DAYS):
    cutoff = datetime.utcnow() - timedelta(days=refresh_days)
    cutoff_iso = cutoff.isoformat()
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT Ticker FROM FilteredTickers WHERE LastChecked >= ?", (cutoff_iso,))
    filtered = {r[0] for r in cur.fetchall()}
    cur.execute("SELECT Ticker FROM RejectedTickers WHERE LastChecked >= ?", (cutoff_iso,))
    rejected = {r[0] for r in cur.fetchall()}
    conn.close()
    return filtered | rejected

# === SAVE (CSV) OPTIONAL ===
def save_checkpoint(data, path):
    pd.DataFrame(data or []).to_csv(path, index=False)
    log(f"💾 Saved {len(data or [])} records to {path}")

def export_checkpoints_from_db(db_path=DB_FILE):
    with sqlite3.connect(db_path) as conn:
        filtered = pd.read_sql("SELECT Ticker, Name, Price FROM FilteredTickers ORDER BY Ticker", conn)
        rejected = pd.read_sql("SELECT Ticker, Reason FROM RejectedTickers ORDER BY Ticker", conn)
    filtered.to_csv(CHECKPOINT_FILE, index=False)
    rejected.to_csv(REJECTED_FILE, index=False)
    log(f"💾 Exported {len(filtered)} filtered and {len(rejected)} rejected checkpoint rows from DB.")

# === SAFE REQUEST ===
def safe_request(url, max_retries=3, delay=2, timeout=15):
    for attempt in range(max_retries):
        try:
            r = SESSION.get(url, headers=HEADERS, timeout=timeout)
            if r.status_code == 429:
                sleep_s = delay * (attempt + 1)
                log(f"⏳ Rate limited. Sleeping {sleep_s}s...")
                time.sleep(sleep_s)
                continue
            r.raise_for_status()
            return r
        except Exception as e:
            log(f"⚠️ Error on request to {url}: {e}")
            # small jitter to avoid thundering herd
            time.sleep(delay * (attempt + 1) + random.uniform(0, 0.5))
    return None

# === HISTORICAL DATA ===
def get_history(ticker):
    # 1-year span to compute liquidity & volatility
    url = f"https://api.robinhood.com/quotes/historicals/{ticker}/?interval=day&span=year"
    r = safe_request(url)
    if not r:
        return [], []
    try:
        data = r.json().get("historicals", []) or []
        closes = []
        volumes = []
        for d in data:
            cp = d.get("close_price")
            vol = d.get("volume")
            if cp is None or cp == "0.0000":
                continue
            try:
                closes.append(float(cp))
                volumes.append(int(vol) if vol is not None else 0)
            except Exception:
                # skip malformed rows
                continue
        # make sure lengths match
        n = min(len(closes), len(volumes))
        return closes[:n], volumes[:n]
    except Exception:
        return [], []

# === QUOTE CHECK ===
def is_valid_quote(ticker):
    url = f"https://api.robinhood.com/quotes/{ticker}/"
    r = safe_request(url)
    if not r:
        return False
    try:
        data = r.json()
        return data.get("last_trade_price") is not None
    except Exception:
        return False

# === FILTERS ===
def check_liquidity(closes, volumes):
    if len(closes) < 3:
        return False, None, None
    try:
        avg_vol = sum(volumes) / len(volumes)
        avg_dollar_vol = sum(c * v for c, v in zip(closes, volumes)) / len(closes)
    except Exception:
        return False, None, None
    ok = (avg_vol >= MIN_VOLUME) and (avg_dollar_vol >= MIN_DOLLAR_VOLUME)
    return ok, avg_vol, avg_dollar_vol

def check_volatility(closes):
    if len(closes) < 3:
        return False
    returns = pd.Series(closes).pct_change().dropna()
    return (not returns.empty) and (returns.std() <= MAX_STDDEV)

def check_price(closes):
    return bool(closes) and closes[-1] >= MIN_PRICE

def check_momentum(closes):
    if len(closes) < 252:
        return False
    base = closes[-252]
    if base == 0:
        return False
    one_year_return = (closes[-1] - base) / base
    return one_year_return >= 0.05

def trade_freq_ok(avg_vol):
    if not USE_TRADE_FREQ_FILTER:
        return True, None
    trades_per_min = (avg_vol / max(1, ASSUMED_AVG_TRADE_SIZE)) / SESSION_MINUTES
    return trades_per_min >= MIN_TRADES_PER_MIN, trades_per_min

# === REASON TRACKER ===
reason_counter = Counter()

# === PROCESS TICKER ===
def process_ticker(item):
    ticker = item["symbol"]
    name = item.get("name", "") or ""
    try:
        if not is_valid_quote(ticker):
            reason_counter["Invalid quote"] += 1
            upsert_rejected(ticker, "Invalid quote")
            return None, {"Ticker": ticker, "Reason": "Invalid quote"}

        closes, volumes = get_history(ticker)
        if not closes or not volumes:
            reason_counter["No history"] += 1
            upsert_rejected(ticker, "No history")
            return None, {"Ticker": ticker, "Reason": "No history"}

        if not check_price(closes):
            reason_counter["Price < $3"] += 1
            upsert_rejected(ticker, "Price < $3")
            return None, {"Ticker": ticker, "Reason": "Price < $3"}

        liq_ok, avg_vol, avg_dollar_vol = check_liquidity(closes, volumes)
        if not liq_ok:
            reason_counter["Liquidity fail"] += 1
            upsert_rejected(ticker, "Liquidity fail")
            return None, {"Ticker": ticker, "Reason": "Liquidity fail"}

        tf_ok, trades_per_min = trade_freq_ok(avg_vol)
        if not tf_ok:
            reason_counter["Trade frequency fail"] += 1
            upsert_rejected(ticker, f"Trade frequency < {MIN_TRADES_PER_MIN}/min (est)")
            return None, {"Ticker": ticker, "Reason": "Trade frequency fail"}

        if not check_volatility(closes):
            reason_counter["Volatility fail"] += 1
            upsert_rejected(ticker, "Volatility fail")
            return None, {"Ticker": ticker, "Reason": "Volatility fail"}

        # Momentum optional (currently disabled)
        # if not check_momentum(closes):
        #     reason_counter["No uptrend"] += 1
        #     upsert_rejected(ticker, "No uptrend")
        #     return None, {"Ticker": ticker, "Reason": "No uptrend"}

        result = {
            "Ticker": ticker,
            "Name": name,
            "Price": closes[-1],
            "LastChecked": datetime.utcnow().isoformat()
        }
        upsert_filtered(result)
        log(f"✅ {ticker} passed all filters.")
        return result, None

    except Exception as e:
        reason_counter["Processing exception"] += 1
        upsert_rejected(ticker, "Processing exception")
        log(f"❌ Error processing {ticker}: {e}")
        return None, {"Ticker": ticker, "Reason": "Processing exception"}

# === CHECKPOINT LOADER (kept for continuity/CSV audit) ===
def read_checkpoint(path, label):
    try:
        if os.path.exists(path) and os.path.getsize(path) > 0:
            df = pd.read_csv(path)
            if not df.empty:
                log(f"📂 Loaded {len(df)} {label} from {path}")
                return df.to_dict('records')
    except Exception:
        pass
    return []

# === MAIN ===
def main():
    init_db()

    # Load skip set based on freshness window
    skip_recent = load_recently_checked(db_path=DB_FILE, refresh_days=REFRESH_DAYS)

    # Keep CSV checkpoints for human auditing (optional)
    final_filtered_csv = read_checkpoint(CHECKPOINT_FILE, "filtered tickers")
    rejected_csv = read_checkpoint(REJECTED_FILE, "rejected tickers")

    if RETRY_FAILED_ONLY:
        tickers_to_process = [{"symbol": t["Ticker"], "name": ""} for t in rejected_csv]
        log(f"🔁 Retrying {len(tickers_to_process)} previously rejected tickers (ignoring freshness window)...")
    else:
        url = "https://api.robinhood.com/instruments/"
        next_url = url
        tickers_to_process = []
        log("📥 Gathering tradable stock tickers...")
        while next_url:
            r = safe_request(next_url)
            if not r:
                break
            data = r.json()
            for item in data.get("results", []):
                if item.get("type") == "stock" and item.get("tradeable"):
                    sym = item.get("symbol")
                    if not sym:
                        continue
                    if sym not in skip_recent:  # skip if recently checked
                        tickers_to_process.append({"symbol": sym, "name": item.get("name","")})
            next_url = data.get("next")

    log(f"🔍 Filtering {len(tickers_to_process)} tickers (skipping {len(skip_recent)} fresh tickers)...")

    final_filtered = []
    rejected_tickers = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_ticker, item): item["symbol"] for item in tickers_to_process}
        for idx, future in enumerate(tqdm(as_completed(futures), total=len(futures), desc="Processing")):
            ticker = futures[future]
            try:
                result, error = future.result()
                if result:
                    final_filtered.append(result)
                elif error:
                    rejected_tickers.append(error)
            except Exception:
                reason_counter["Unhandled exception"] += 1
                upsert_rejected(ticker, "Unhandled exception")
                rejected_tickers.append({"Ticker": ticker, "Reason": "Unhandled exception"})

            if (len(final_filtered) + len(rejected_tickers)) % SAVE_INTERVAL == 0:
                # drop LastChecked in CSV for compactness
                slim = [{k:v for k,v in row.items() if k != "LastChecked"} for row in final_filtered]
                save_checkpoint(slim, CHECKPOINT_FILE)
                save_checkpoint(rejected_tickers, REJECTED_FILE)

            time.sleep(RATE_LIMIT_DELAY)

    # Final CSV audit export includes cached rows skipped by incremental refresh.
    export_checkpoints_from_db()

    log(f"✅ Done. {len(final_filtered)} updated/added, {len(rejected_tickers)} rejected/updated.")
    log("📊 Rejection reasons summary:")
    for reason, count in reason_counter.most_common():
        log(f"{reason}: {count}")

if __name__ == "__main__":
    main()


In [ ]:
import os, sqlite3, time, requests
import numpy as np, pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta

# === Paths ===
BASE_DIR = str(PROJECT_ROOT)
TICKER_DB_PATH = os.path.join(BASE_DIR, "filtered_tickers.db")
HIST_DB_PATH   = os.path.join(BASE_DIR, "historicals.db")
VEC_DB_PATH    = os.path.join(BASE_DIR, "vectorized.db")

# === Config ===
DT_FMT = "%Y-%m-%d %H:%M:%S"
def dtstr(dt):
    if dt is None or pd.isna(dt): return None
    if isinstance(dt, pd.Timestamp):
        dt = dt.to_pydatetime()
    return dt.strftime(DT_FMT)

FEATURES_FOR_SIM = [
    "close_price","volume",
    "pct_1d","pct_2d","pct_3d","pct_5d",
    "ret_10d","ret_20d","ret_60d",
    "volatility_5d","volatility_10d","vol_20d","vol_60d",
    "trend_slope_60d","trend_r2_60d",
    "riskadj_mom_60d",
    "ma_5d","ma_20d","ma_crossover","z_ma20","bb_width_20d",
    "dollar_vol_20d","ac1_5d","max_dd_60d","time_since_max_60d",
]
NEW_COLS = {
    "ret_10d":"REAL","ret_20d":"REAL","ret_60d":"REAL",
    "riskadj_mom_60d":"REAL",
    "vol_20d":"REAL","vol_60d":"REAL",
    "trend_slope_60d":"REAL","trend_r2_60d":"REAL",
    "z_ma20":"REAL","bb_width_20d":"REAL",
    "dollar_vol_20d":"REAL","ac1_5d":"REAL",
    "max_dd_60d":"REAL","time_since_max_60d":"REAL",
}

ROLLING_MAX_WINDOW = 120
SPANS    = ["week","month","3month","year","5year"]
INTERVAL = "day"
MAX_RETRIES, RETRY_DELAY = 3, 3
DO_BACKFILL = True
REQUEST_HEADERS = globals().get("HEADERS", {})
EMPTY_HISTORY_COUNT = 0
EMPTY_HISTORY_SAMPLES = []

# span retention windows (approximate, trading days not required)
SPAN_DAYS = {"week": 7, "month": 31, "3month": 95, "year": 366, "5year": 5*366}

# --- helpers ------------------------------------------------------------------
def load_filtered_tickers(db_path):
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql("SELECT Ticker FROM FilteredTickers", conn)
    return df["Ticker"].dropna().unique().tolist()

def compute_features(df):  # df indexed by begins_at; has close_price & volume
    out = df.copy()
    r = out["close_price"].pct_change(1)

    # existing
    out["pct_1d"] = r
    out["pct_2d"] = out["close_price"].pct_change(2)
    out["pct_3d"] = out["close_price"].pct_change(3)
    out["pct_5d"] = out["close_price"].pct_change(5)
    out["volatility_5d"]  = r.rolling(5).std()
    out["volatility_10d"] = r.rolling(10).std()
    out["ma_5d"]  = out["close_price"].rolling(5).mean()
    out["ma_20d"] = out["close_price"].rolling(20).mean()
    out["ma_crossover"] = (out["ma_5d"] > out["ma_20d"]).astype(int)

    # new
    out["ret_10d"] = out["close_price"].pct_change(10)
    out["ret_20d"] = out["close_price"].pct_change(20)
    out["ret_60d"] = out["close_price"].pct_change(60)
    out["vol_20d"] = r.rolling(20).std()
    out["vol_60d"] = r.rolling(60).std()
    out["riskadj_mom_60d"] = out["ret_60d"] / out["vol_60d"]

    std20 = out["close_price"].rolling(20).std()
    out["z_ma20"] = (out["close_price"] - out["ma_20d"]) / std20
    out["bb_width_20d"] = (4 * std20) / out["ma_20d"]  # k=2 bands

    def _slope_60(x):
        if np.std(x)==0: return np.nan
        return np.polyfit(np.arange(60, dtype=float), x, 1)[0]
    def _r2_60(x):
        if np.std(x)==0: return np.nan
        r = np.corrcoef(np.arange(60, dtype=float), x)[0,1]
        return r*r
    out["trend_slope_60d"] = out["close_price"].rolling(60).apply(_slope_60, raw=True)
    out["trend_r2_60d"]    = out["close_price"].rolling(60).apply(_r2_60, raw=True)

    out["dollar_vol_20d"] = (out["close_price"]*out["volume"]).rolling(20).mean()
    def _acf1(a):
        a = np.asarray(a, float)
        if len(a)<2 or not np.isfinite(a).all(): return np.nan
        x0, x1 = a[:-1], a[1:]
        sx, sy = x0.std(ddof=1), x1.std(ddof=1)
        if sx==0 or sy==0: return np.nan
        return float(np.corrcoef(x0, x1)[0,1])
    out["ac1_5d"] = r.rolling(5).apply(_acf1, raw=False)

    def _max_dd_np(a):
        peak = -np.inf; mdd = 0.0
        for v in a:
            if v > peak: peak = v
            if peak > 0:
                dd = (v/peak) - 1.0
                if dd < mdd: mdd = dd
        return mdd
    out["max_dd_60d"] = out["close_price"].rolling(60).apply(_max_dd_np, raw=True)
    out["time_since_max_60d"] = out["close_price"].rolling(60).apply(
        lambda x: len(x)-1 - int(np.argmax(x)), raw=True
    )
    return out

def ensure_schema_and_indexes():
    os.makedirs(BASE_DIR, exist_ok=True)
    ch = sqlite3.connect(HIST_DB_PATH)
    cv = sqlite3.connect(VEC_DB_PATH)

    ch.execute("""
    CREATE TABLE IF NOT EXISTS HistoricalPrices (
      begins_at TEXT, open_price REAL, close_price REAL, high_price REAL, low_price REAL,
      volume INTEGER, session TEXT, interpolated INTEGER, ticker TEXT, span TEXT, pulled_at TEXT
    )""")
    cv.execute("""
    CREATE TABLE IF NOT EXISTS VectorizedFeatures (
      begins_at TEXT,
      open_price REAL, close_price REAL, high_price REAL, low_price REAL, volume INTEGER,
      session TEXT, interpolated INTEGER, ticker TEXT, span TEXT, pulled_at TEXT,
      pct_1d REAL, pct_2d REAL, pct_3d REAL, pct_5d REAL,
      volatility_5d REAL, volatility_10d REAL,
      momentum_slope_5d REAL,
      ma_5d REAL, ma_20d REAL, ma_crossover INTEGER
    )""")
    ch.commit(); cv.commit()

    # de-dupe existing rows so unique indexes can be (and stay) valid
    ch.execute("""DELETE FROM HistoricalPrices
                  WHERE rowid NOT IN (
                    SELECT MIN(rowid) FROM HistoricalPrices
                    GROUP BY ticker, span, begins_at
                  )""")
    cv.execute("""DELETE FROM VectorizedFeatures
                  WHERE rowid NOT IN (
                    SELECT MIN(rowid) FROM VectorizedFeatures
                    GROUP BY ticker, span, begins_at
                  )""")
    ch.commit(); cv.commit()

    ch.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_hp ON HistoricalPrices(ticker,span,begins_at)")
    cv.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_vf ON VectorizedFeatures(ticker,span,begins_at)")
    ch.execute("CREATE INDEX IF NOT EXISTS idx_hp_span_date ON HistoricalPrices(span,begins_at)")
    cv.execute("CREATE INDEX IF NOT EXISTS idx_vf_span_date ON VectorizedFeatures(span,begins_at)")

    # add new columns if missing
    existing_vf = {r[1] for r in cv.execute("PRAGMA table_info(VectorizedFeatures)")}
    for col, typ in NEW_COLS.items():
        if col not in existing_vf:
            cv.execute(f"ALTER TABLE VectorizedFeatures ADD COLUMN {col} {typ}")
    cv.commit(); ch.commit(); ch.close(); cv.close()

def backfill_new_columns():
    if not DO_BACKFILL:
        return
    vconn = sqlite3.connect(VEC_DB_PATH); hconn = sqlite3.connect(HIST_DB_PATH)

    pairs = pd.read_sql("SELECT DISTINCT ticker, span FROM VectorizedFeatures", vconn)
    need_expr = " OR ".join([f"{c} IS NULL" for c in NEW_COLS.keys()])

    for _, row in tqdm(pairs.iterrows(), total=len(pairs), desc="Backfilling"):
        t, s = row["ticker"], row["span"]
        need = pd.read_sql(
            f"""SELECT begins_at FROM VectorizedFeatures
                WHERE ticker=? AND span=? AND ({need_expr})
                ORDER BY begins_at""",
            vconn, params=(t, s), parse_dates=["begins_at"]
        )
        if need.empty:
            continue

        first_needed = need["begins_at"].min()

        hist = pd.read_sql(
            """SELECT begins_at, close_price, volume
               FROM HistoricalPrices
               WHERE ticker=? AND span=? AND begins_at <= ?
               ORDER BY begins_at""",
            hconn,
            params=(t, s, dtstr(first_needed)),
            parse_dates=["begins_at"]
        )
        extra = pd.read_sql(
            """SELECT begins_at, close_price, volume
               FROM HistoricalPrices
               WHERE ticker=? AND span=? AND begins_at > ?
               ORDER BY begins_at""",
            hconn,
            params=(t, s, dtstr(first_needed)),
            parse_dates=["begins_at"]
        )

        prices = pd.concat([hist.tail(ROLLING_MAX_WINDOW), extra], ignore_index=True)
        if prices.empty:
            continue

        prices = prices.set_index("begins_at").sort_index()
        feats = compute_features(prices).reset_index()
        upd = feats[feats["begins_at"].isin(need["begins_at"])]

        for _, r in upd.iterrows():
            params = [None if pd.isna(r[c]) else float(r[c]) for c in NEW_COLS.keys()]
            params += [t, s, dtstr(r["begins_at"])]
            vconn.execute(f"""
                UPDATE VectorizedFeatures SET
                {', '.join([f'{c}=?' for c in NEW_COLS.keys()])}
                WHERE ticker=? AND span=? AND begins_at=?;
            """, params)

    vconn.commit(); vconn.close(); hconn.close()

def prune_out_of_window(conn, table, span, now=None):
    """Keep only rows within the intended span window (e.g., 5year keeps ~5y)."""
    days = SPAN_DAYS[span]
    now = now or datetime.utcnow()
    cutoff = now - timedelta(days=days)
    cur = conn.cursor()
    cur.execute(
        f"DELETE FROM {table} WHERE span=? AND begins_at < ?",
        (span, dtstr(cutoff))
    )
    conn.commit()

# ---------- main updater ----------
def update_historicals_and_vectorized(tickers, interval=INTERVAL):
    global EMPTY_HISTORY_COUNT, EMPTY_HISTORY_SAMPLES
    ensure_schema_and_indexes()
    if DO_BACKFILL:
        backfill_new_columns()

    ch = sqlite3.connect(HIST_DB_PATH)
    cv = sqlite3.connect(VEC_DB_PATH)

    # last vectorized date per (ticker,span)
    try:
        last_vec_df = pd.read_sql(
            """SELECT ticker, span, MAX(begins_at) AS last_vec
               FROM VectorizedFeatures GROUP BY ticker, span""",
            cv,
            parse_dates=["last_vec"]
        )
        last_vec = {(r["ticker"], r["span"]): (pd.NaT if pd.isna(r["last_vec"]) else r["last_vec"].to_pydatetime())
                    for _, r in last_vec_df.iterrows() if r["last_vec"] is not None}
    except Exception:
        last_vec = {}

    # prune old rows for every span once at the start (keeps DB bounded)
    for s in SPANS:
        prune_out_of_window(ch, "HistoricalPrices", s)
        prune_out_of_window(cv, "VectorizedFeatures", s)

    for ticker in tqdm(tickers, desc="Updating"):
        for span in SPANS:
            # last historical dt
            row = pd.read_sql(
                """SELECT MAX(begins_at) AS last_hist
                   FROM HistoricalPrices WHERE ticker=? AND span=?""",
                ch, params=(ticker, span), parse_dates=["last_hist"]
            )
            last_hist = row.iloc[0,0]
            if pd.notna(last_hist):
                last_hist = last_hist.to_pydatetime()

            success = False
            for attempt in range(MAX_RETRIES):
                try:
                    url = f"https://api.robinhood.com/quotes/historicals/{ticker}/?interval={interval}&span={span}"
                    r = requests.get(url, headers=REQUEST_HEADERS, timeout=20)
                    if r.status_code == 429:
                        time.sleep(RETRY_DELAY * (attempt+1))
                        continue
                    r.raise_for_status()
                    hist = r.json().get("historicals", [])
                    if not hist:
                        EMPTY_HISTORY_COUNT += 1
                        if len(EMPTY_HISTORY_SAMPLES) < 10:
                            EMPTY_HISTORY_SAMPLES.append(f"{ticker}-{span} status={r.status_code} body={r.text[:240]}")
                        success = True; break

                    df_new = pd.DataFrame(hist)
                    for col in ["open_price","close_price","high_price","low_price","volume"]:
                        df_new[col] = pd.to_numeric(df_new[col], errors="coerce")
                    df_new["begins_at"] = pd.to_datetime(df_new["begins_at"]).dt.tz_localize(None)

                    if pd.notna(last_hist):
                        df_new = df_new[df_new["begins_at"] > last_hist]
                    if df_new.empty:
                        success = True; break

                    df_new = df_new.sort_values("begins_at").drop_duplicates(subset=["begins_at"])
                    df_new["ticker"] = ticker
                    df_new["span"] = span
                    df_new["pulled_at"] = pd.Timestamp.utcnow()

                    # --- anti-join against existing HP keys ---
                    existing_hp = pd.read_sql(
                        """SELECT begins_at FROM HistoricalPrices
                           WHERE ticker=? AND span=? AND begins_at >= ? AND begins_at <= ?""",
                        ch,
                        params=(ticker, span,
                                dtstr(df_new["begins_at"].min()),
                                dtstr(df_new["begins_at"].max())),
                        parse_dates=["begins_at"]
                    )
                    if not existing_hp.empty:
                        df_new = df_new[~df_new["begins_at"].isin(existing_hp["begins_at"])]
                    if df_new.empty:
                        success = True; break

                    df_new.to_sql("HistoricalPrices", ch, if_exists="append", index=False)

                    # ---- vectorize with context (rolling windows need tail)
                    min_new_dt = df_new["begins_at"].min()
                    tail = pd.read_sql(
                        """SELECT begins_at, close_price, volume
                           FROM HistoricalPrices
                           WHERE ticker=? AND span=? AND begins_at < ?
                           ORDER BY begins_at DESC LIMIT ?""",
                        ch,
                        params=(ticker, span, dtstr(min_new_dt), ROLLING_MAX_WINDOW),
                        parse_dates=["begins_at"]
                    )
                    base = (pd.concat([tail.sort_values("begins_at"),
                                       df_new[["begins_at","close_price","volume"]]], ignore_index=True)
                            if not tail.empty else df_new[["begins_at","close_price","volume"]].copy())
                    base = base.set_index("begins_at").sort_index()
                    feats = compute_features(base)

                    lv = last_vec.get((ticker, span))
                    if isinstance(lv, pd.Timestamp):
                        lv = lv.to_pydatetime()
                    if lv is not None and not pd.isna(lv):
                        feats = feats[feats.index > lv]
                    else:
                        feats = feats[feats.index >= min_new_dt]

                    if not feats.empty:
                        out = feats.reset_index()
                        # carry optional fields if present
                        out = out.merge(df_new[["begins_at","session","interpolated"]], on="begins_at", how="left")
                        out["ticker"] = ticker
                        out["span"] = span
                        out["pulled_at"] = pd.Timestamp.utcnow()
                        out = out.drop_duplicates(subset=["ticker","span","begins_at"])

                        # --- anti-join against existing VF keys ---
                        existing_vf = pd.read_sql(
                            """SELECT begins_at FROM VectorizedFeatures
                               WHERE ticker=? AND span=? AND begins_at >= ? AND begins_at <= ?""",
                            cv,
                            params=(ticker, span,
                                    dtstr(out["begins_at"].min()),
                                    dtstr(out["begins_at"].max())),
                            parse_dates=["begins_at"]
                        )
                        if not existing_vf.empty:
                            out = out[~out["begins_at"].isin(existing_vf["begins_at"])]

                        if not out.empty:
                            out.to_sql("VectorizedFeatures", cv, if_exists="append", index=False)

                    # keep DB bounded for this span (post-append prune)
                    prune_out_of_window(ch, "HistoricalPrices", span)
                    prune_out_of_window(cv, "VectorizedFeatures", span)

                    success = True; break
                except Exception as e:
                    print(f"❌ {ticker}-{span} try {attempt+1}: {e}")
          
                    
                    
                    
                    
                    
                    
                    
                    
                            
                    time.sleep(RETRY_DELAY)

            if not success:
                print(f"⚠️ Gave up on {ticker}-{span}")

    ch.close(); cv.close()
    if EMPTY_HISTORY_COUNT:
        print(f"⚠️ Empty Robinhood historical responses: {EMPTY_HISTORY_COUNT}")
        for sample in EMPTY_HISTORY_SAMPLES:
            print(f"  {sample}")
    print("✅ Update complete")

# ---------- RUN ----------
def report_required_app_data():
    required_spans = ["year", "5year"]
    problems = []
    with sqlite3.connect(HIST_DB_PATH) as hconn, sqlite3.connect(VEC_DB_PATH) as vconn:
        for db_label, conn, table in [
            ("historicals", hconn, "HistoricalPrices"),
            ("vectorized", vconn, "VectorizedFeatures"),
        ]:
            counts = pd.read_sql(
                f"""SELECT span, COUNT(*) AS rows, MAX(begins_at) AS max_dt,
                          COUNT(DISTINCT ticker) AS tickers
                   FROM {table}
                   GROUP BY span""",
                conn,
            )
            print(f"\n{db_label} app database span coverage ({table}):")
            if counts.empty:
                print("  no rows")
                problems.append(f"{table} has no rows")
                continue
            print(counts.sort_values("span").to_string(index=False))
            present = set(counts.loc[counts["rows"] > 0, "span"].astype(str))
            missing = [span for span in required_spans if span not in present]
            if missing:
                problems.append(f"{table} missing populated spans: {', '.join(missing)}")
    if problems:
        raise RuntimeError("Stock prediction app database is incomplete: " + "; ".join(problems))

if __name__ == "__main__":
    tickers = load_filtered_tickers(TICKER_DB_PATH)
    if not tickers:
        raise RuntimeError(f"No filtered tickers found in {TICKER_DB_PATH}; cannot build app database.")
    update_historicals_and_vectorized(tickers)
    report_required_app_data()
    print("Features ready. Use FEATURES_FOR_SIM downstream for similarity calculations.")


In [ ]:
import matplotlib.pyplot as plt
import sqlite3
import pandas as pd

conn = sqlite3.connect(VEC_DB_PATH)
df = pd.read_sql("""
    SELECT begins_at, ticker, close_price
    FROM VectorizedFeatures
    WHERE span = '5year'
""", conn)
conn.close()

df["begins_at"] = pd.to_datetime(df["begins_at"])
df = df.sort_values(["ticker", "begins_at"]).reset_index(drop=True)

# vectorized normalize per ticker
df["norm_price"] = df.groupby("ticker")["close_price"].transform(lambda s: s / s.iloc[0])

plt.figure(figsize=(16, 8))
for t, g in df.groupby("ticker"):
    plt.plot(g["begins_at"], g["norm_price"], alpha=0.15, linewidth=0.8)

plt.title("📈 Normalized Price Trends (5y span)")
plt.xlabel("Date"); plt.ylabel("Normalized Close (start=1.0)")
plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np

# === Paths ===
BASE_DIR = str(PROJECT_ROOT)
VEC_DB_PATH = os.path.join(BASE_DIR, "vectorized.db")
HIST_DB_PATH = os.path.join(BASE_DIR, "historicals.db")
FILTERED_DB_PATH = os.path.join(BASE_DIR, "filtered_tickers.db")
OUTPUT_CSV = os.path.join(BASE_DIR, "vector_analysis_results.csv")
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# === Which span to summarize ===
REQUESTED_SPAN = "5year"   # change to 'year', '3month', etc. if you want
SPAN = REQUESTED_SPAN

# === Columns we expect from the migrated vectorizer ===
REQUIRED = [
    "begins_at","ticker","span","close_price","volume",
    "trend_slope_60d","trend_r2_60d",
    "vol_60d","riskadj_mom_60d",
    "dollar_vol_20d","max_dd_60d",
    "ac1_5d","z_ma20","bb_width_20d","ma_crossover"
]

def read_cols(conn, table):
    rows = conn.execute(f"PRAGMA table_info({table})").fetchall()
    return [r[1] for r in rows]

def describe_vectorized_sources():
    roots = []
    for root in [Path(PROJECT_ROOT), Path.cwd(), Path.cwd() / "data", Path.cwd() / "stockprediction2025" / "data", Path(PROJECT_ROOT) / "data"]:
        root = root.resolve()
        if root not in roots:
            roots.append(root)
    lines = []
    for root in roots:
        db = root / "vectorized.db"
        if not db.exists():
            lines.append(f"{db}: missing")
            continue
        try:
            with sqlite3.connect(db) as check_conn:
                stats = pd.read_sql(
                    """SELECT span, COUNT(*) AS rows, MAX(begins_at) AS max_dt
                       FROM VectorizedFeatures GROUP BY span""",
                    check_conn,
                )
        except Exception as exc:
            lines.append(f"{db}: unreadable ({exc})")
            continue
        if stats.empty:
            lines.append(f"{db}: VectorizedFeatures has no rows")
        else:
            summary = ", ".join(f"{r.span}:{int(r.rows)} max={r.max_dt}" for r in stats.itertuples())
            lines.append(f"{db}: {summary}")
    return "\n".join(lines)

# === Load data (only needed columns, single span) ===
conn = sqlite3.connect(VEC_DB_PATH)
have_cols = set(read_cols(conn, "VectorizedFeatures"))
missing = [c for c in REQUIRED if c not in have_cols]
if missing:
    raise RuntimeError(f"VectorizedFeatures missing expected columns: {missing}\n"
                       f"Run the migration/updater cell first, then retry.")

span_counts = pd.read_sql(
    """SELECT span, COUNT(*) AS rows, MAX(begins_at) AS max_dt
       FROM VectorizedFeatures GROUP BY span""",
    conn,
)
if span_counts.empty or not (span_counts["rows"] > 0).any():
    conn.close()
    raise RuntimeError(
        "VectorizedFeatures has no populated spans to summarize. Missing app database data:\n"
        + describe_vectorized_sources()
    )
if not ((span_counts["span"] == REQUESTED_SPAN) & (span_counts["rows"] > 0)).any():
    pref = {"5year": 5, "year": 4, "3month": 3, "month": 2, "week": 1}
    ranked = span_counts[span_counts["rows"] > 0].copy()
    ranked["pref"] = ranked["span"].map(pref).fillna(0)
    ranked = ranked.sort_values(["max_dt", "pref", "rows"], ascending=[False, False, False])
    SPAN = ranked.iloc[0]["span"]
    print(f"Requested span={REQUESTED_SPAN!r} unavailable for summary; using span={SPAN!r}.")

q = f"""
SELECT {", ".join(REQUIRED)}
FROM VectorizedFeatures
WHERE span = ?
"""
df = pd.read_sql(q, conn, params=(SPAN,), parse_dates=["begins_at"])
conn.close()

# Clean & order
df = df.sort_values(["ticker","begins_at"]).reset_index(drop=True)

# === Per-ticker summary ===
def summarize(group: pd.DataFrame) -> pd.Series:
    # price-based totals
    start_px = group["close_price"].iloc[0]
    end_px   = group["close_price"].iloc[-1]
    total_ret = (end_px / start_px - 1.0) if start_px and np.isfinite(start_px) else np.nan

    # Use the newest usable metric even if the final appended row is sparse.
    def last_valid(column):
        values = group[column].dropna()
        return values.iloc[-1] if not values.empty else np.nan

    # simple quality/lead scores you can sort on
    trend_slope   = last_valid("trend_slope_60d")
    trend_quality = last_valid("trend_r2_60d")
    vol_60d       = last_valid("vol_60d")
    trend_score   = (trend_slope / vol_60d) if np.isfinite(vol_60d) and vol_60d else np.nan
    leader_score  = last_valid("riskadj_mom_60d")  # already risk-adjusted momentum

    return pd.Series({
        "Start": group["begins_at"].iloc[0],
        "End":   group["begins_at"].iloc[-1],
        "Rows":  len(group),
        "Total_Return": total_ret,
        "Trend_Slope_60d": trend_slope,
        "Trend_R2_60d":    trend_quality,
        "Vol_60d":         vol_60d,
        "RiskAdj_Mom_60d": leader_score,
        "DollarVol_20d":   last_valid("dollar_vol_20d"),
        "MaxDD_60d":       last_valid("max_dd_60d"),
        "AC1_5d":          last_valid("ac1_5d"),
        "Z_MA20":          last_valid("z_ma20"),
        "BB_Width_20d":    last_valid("bb_width_20d"),
        "MA_Crossover":    int(last_valid("ma_crossover")) if pd.notna(last_valid("ma_crossover")) else None,
        "Trend_Score":     trend_score,   # slope / vol
        "Leader_Score":    leader_score   # alias for sorting
    })

feature_df = (
    df.groupby("ticker", as_index=True)
      .apply(summarize)
      .reset_index()
)

# Keep the analysis output limited to tickers with usable rolling metrics.
before_required_metrics = len(feature_df)
feature_df = feature_df.dropna(subset=["Leader_Score", "Trend_Slope_60d"])
dropped_required_metrics = before_required_metrics - len(feature_df)
if dropped_required_metrics:
    print(f"Skipped {dropped_required_metrics} tickers without usable 60-day summary metrics.")

# Guard against empty spans / schema surprises before sorting
rank_cols = [c for c in ["Leader_Score","Trend_Score","DollarVol_20d"] if c in feature_df.columns]
if rank_cols and not feature_df.empty:
    feature_df = feature_df.sort_values(rank_cols, ascending=[False] * len(rank_cols))

# Save to DB and CSV
conn = sqlite3.connect(VEC_DB_PATH)
feature_df.to_sql("FeatureSummary", conn, if_exists="replace", index=False)
conn.close()
feature_df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Saved FeatureSummary ({len(feature_df)} tickers) → DB:{VEC_DB_PATH} table=FeatureSummary")
print(f"💾 Also wrote CSV → {OUTPUT_CSV}")

# Optional: quick peek at the top 15 “leaders”
display_cols = ["ticker","Leader_Score","Trend_Score","Trend_R2_60d","Vol_60d","DollarVol_20d","MaxDD_60d","Total_Return","End"]
avail_cols = [c for c in display_cols if c in feature_df.columns]
print("\nTop 15 by Leader_Score:")
if feature_df.empty:
    print("(no rows returned for selected span)")
elif avail_cols:
    print(feature_df[avail_cols].head(15).to_string(index=False))
else:
    print("(no display columns available)")


In [ ]:
import os, re, sqlite3, math
from pathlib import Path
import numpy as np, pandas as pd
from tqdm import tqdm

# ==== knobs ====
SPAN = "5year"

# ==== paths ====
# Some IDE/GitHub runners start in different directories or keep stale empty DBs.
# Prefer the requested span, then fall back to the newest populated span so the
# action can finish with the best available data instead of failing late.
def _candidate_roots():
    roots = []
    if "PROJECT_ROOT" in globals():
        roots.append(Path(PROJECT_ROOT))
    cwd = Path.cwd()
    roots.extend([
        cwd,
        cwd.parent,
        cwd / "stockprediction2025" / "data",
        cwd / "data",
        cwd.parent / "stockprediction2025" / "data",
        cwd.parent / "data",
    ])
    seen = set()
    for root in roots:
        root = root.resolve()
        if root not in seen:
            seen.add(root)
            yield root

def _table_span_stats(db_path, table):
    if not db_path.exists():
        return None
    try:
        with sqlite3.connect(db_path) as conn:
            sql = f"""SELECT span, COUNT(*) AS rows, MAX(begins_at) AS max_dt
                      FROM {table}
                      GROUP BY span"""
            return pd.read_sql(sql, conn)
    except Exception:
        return None

def _resolve_data_root(requested_span, db_name="vectorized.db", table="VectorizedFeatures"):
    checked = []
    candidates = []
    pref = {"5year": 5, "year": 4, "3month": 3, "month": 2, "week": 1}
    for root in _candidate_roots():
        db_path = root / db_name
        stats = _table_span_stats(db_path, table)
        if stats is None or stats.empty:
            checked.append(f"{db_path}:{table} -> unusable/empty")
            continue
        summary = ", ".join(f"{r.span}:{int(r.rows)}@{r.max_dt}" for r in stats.itertuples())
        checked.append(f"{db_path}:{table} -> {summary}")
        for r in stats.itertuples():
            if int(r.rows) > 0:
                exact = 1 if r.span == requested_span else 0
                candidates.append((exact, str(r.max_dt or ""), pref.get(r.span, 0), int(r.rows), root, r.span))
    if candidates:
        exact, max_dt, pref_score, rows, root, chosen_span = max(candidates)
        if chosen_span != requested_span:
            print(f"Requested span={requested_span!r} is unavailable; using span={chosen_span!r} from {root}.")
        return root, chosen_span, rows, max_dt
    detail = "\n".join(checked)
    raise RuntimeError(f"No populated {table} found for any span. Checked:\n{detail}")

DATA_ROOT, SPAN, _span_rows, _span_max_dt = _resolve_data_root(SPAN, "vectorized.db", "VectorizedFeatures")
BASE_DIR = str(DATA_ROOT)
HIST_DB_PATH = os.path.join(BASE_DIR, "historicals.db")
VEC_DB_PATH  = os.path.join(BASE_DIR, "vectorized.db")
OUT_DIR = os.path.join(BASE_DIR, "analytics"); os.makedirs(OUT_DIR, exist_ok=True)
print(f"Using data root: {BASE_DIR} ({_span_rows:,} VectorizedFeatures rows for span={SPAN}, max={_span_max_dt})")
LOOKBACK_DAYS = 252 * 2
MIN_OVERLAP   = 150
SIM_MIN       = 0.60
SIM_CAP       = 0.88
TOP_N_PER_TICKER = 3
TRY_PLOTLY = True

# Variant-aware family dedup
ENABLE_VARIANT_DEDUP      = True
REQUIRE_BEHAVIOR_CONFIRM  = True   # require family members to be behaviorally similar?
FAMILY_FLIPCORR_MIN       = 0.40   # confirm similarity on sign flips inside a family
KEEP_ONLY_USD_LINE        = True   # prefer non-exchange-suffixed symbol if both exist

# Use features from VectorizedFeatures (fast). If False, fallback builds pct_1d from HistoricalPrices.
USE_VECTORIZED = True

# ==== helpers for variant-aware grouping ====
_EXCH_SUFFIXES = {
    ".TO",".V",".CN",".NE"," .HK".strip(),".SS",".SZ",".L",".DE",".F",".BE",".SW",".PA",".AS",".MI",
    ".BR",".CO",".HE",".OL",".MC",".VI",".VX",".SG",".KS",".KQ",".ST",".IS",".TW",".TWO"
}
_WARRANT_UNIT_PATTERNS = (r"-WS$", r"-WT$", r"-W$", r"-U$", r"\.WS$", r"\.WT$", r"\.U$", r"/WS$", r"/WT$", r"/U$")
_SHARECLASS_PATTERNS    = (r"\.[A-Z]$", r"-[A-Z]$", r"\.PR[A-Z]$", r"-PR[A-Z]$", r"\.P[A-Z]$", r"-P[A-Z]$")

def _is_warrant_or_unit(sym:str) -> bool:
    s = sym.upper()
    return any(re.search(p, s) for p in _WARRANT_UNIT_PATTERNS)

def _strip_exchange_suffix(sym:str) -> str:
    s = sym.upper()
    # remove known exchange/country suffixes (e.g., .TO, .L, .F)
    for suf in sorted(_EXCH_SUFFIXES, key=len, reverse=True):
        if s.endswith(suf):
            return s[: -len(suf)]
    return s

def _strip_shareclass_suffix(s:str) -> str:
    # e.g., BRK.B -> BRK ; XYZ- A -> XYZ ; ABC.PRA -> ABC
    out = s
    for p in _SHARECLASS_PATTERNS:
        out = re.sub(p, "", out)
    return out

def _canonical_base(sym:str) -> str:
    s = sym.upper().strip()
    s = _strip_exchange_suffix(s)
    s = _strip_shareclass_suffix(s)
    # normalize separators
    s = s.replace(".", "").replace("-", "").replace("/", "")
    return s

def _merge_near_bases(bases:list) -> dict:
    """
    Merge bases where one is a prefix of the other with Δlen ≤ 1 (e.g., GOOG ~ GOOGL).
    Returns map: base -> canonical_merged_base.
    """
    bases = sorted(set(bases))
    parent = {b:b for b in bases}
    def find(x):
        while parent[x] != x: x = parent[x]
        return x
    for i, b in enumerate(bases):
        for j in range(i+1, len(bases)):
            c = bases[j]
            if b==c: continue
            if b.startswith(c) and len(b)-len(c) <= 1:
                parent[find(b)] = find(c)
            elif c.startswith(b) and len(c)-len(b) <= 1:
                parent[find(c)] = find(b)
    # compress
    return {b: find(b) for b in bases}

def _family_score_frame(snap: pd.DataFrame, members:list) -> pd.DataFrame:
    sub = snap.loc[members, ["dollar_vol_20d","trend_slope_60d","ret_60d","vol_60d"]].copy()
    # fill NaNs safely
    for c in ["dollar_vol_20d","trend_slope_60d","ret_60d","vol_60d"]:
        if c not in sub.columns: sub[c] = np.nan
    # ranks (higher better), except vol where lower is better
    r_dv  = sub["dollar_vol_20d"].rank(pct=True, na_option="keep")
    r_s   = sub["trend_slope_60d"].rank(pct=True, na_option="keep")
    r_m   = sub["ret_60d"].rank(pct=True, na_option="keep")
    r_v   = (1 - sub["vol_60d"].rank(pct=True, na_option="keep"))  # inverse
    score = 0.40*r_dv.fillna(0.5) + 0.30*r_s.fillna(0.5) + 0.20*r_m.fillna(0.5) + 0.10*r_v.fillna(0.5)
    sub["__score"] = score
    return sub.sort_values("__score", ascending=False)

def _variant_dedup(mat: pd.DataFrame, snap: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns (mat_filtered, snap_filtered, families_df)
    families_df columns: [Family, Member, Chosen, Reason]
    """
    tickers = list(mat.columns)
    # 1) drop warrants/units outright
    drop_wu = [t for t in tickers if _is_warrant_or_unit(t)]
    keep    = [t for t in tickers if t not in drop_wu]

    # 2) compute bases & merge near-bases (GOOG ~ GOOGL)
    base0 = {t: _canonical_base(t) for t in keep}
    merge_map = _merge_near_bases(list(base0.values()))
    base = {t: merge_map.get(base0[t], base0[t]) for t in keep}

    # 3) optionally prefer USD line if duplicated with exchange suffix present
    #    (we already stripped suffixes in the base; USD preference is implicit via base collisions)
    fam = {}
    for t in keep:
        fam.setdefault(base[t], []).append(t)

    # 4) optional behavioral confirmation inside each family (flip-corr among members)
    def _confirm_family(members):
        if not REQUIRE_BEHAVIOR_CONFIRM or len(members) <= 1:
            return members
        # compute small corr matrix between columns for these members only
        cols = [c for c in members if c in mat.columns]
        if len(cols) <= 1:
            return members
        sub = mat[cols]
        good = set([cols[0]])  # seed
        for c in cols[1:]:
            # corr with any already accepted member
            ok = False
            for g in list(good):
                x = sub[g].to_numpy(dtype=float)
                y = sub[c].to_numpy(dtype=float)
                mask = ~(np.isnan(x) | np.isnan(y))
                if mask.sum() >= MIN_OVERLAP:
                    r = np.corrcoef(x[mask], y[mask])[0,1]
                    if np.isfinite(r) and abs(r) >= FAMILY_FLIPCORR_MIN:
                        ok = True; break
            if ok:
                good.add(c)
        # if nothing passed, return original members (be lenient)
        return list(good) if len(good)>=1 else members

    confirmed_fam = {k: _confirm_family(v) for k, v in fam.items()}

    # 5) pick a winner per family by score
    families_rows = []
    chosen = []
    for k, members in confirmed_fam.items():
        if len(members) == 1:
            m = members[0]
            chosen.append(m)
            families_rows.append({"Family": k, "Member": m, "Chosen": True, "Reason": "only member"})
            continue
        rankdf = _family_score_frame(snap, members)
        win = rankdf.index[0]
        chosen.append(win)
        for m in members:
            families_rows.append({
                "Family": k, "Member": m, "Chosen": bool(m==win),
                "Reason": "best score" if m==win else "variant"
            })

    families_df = pd.DataFrame(families_rows).sort_values(["Family","Chosen"], ascending=[True, False])

    # 6) filter matrices
    chosen = sorted(set(chosen))
    mat_f  = mat[chosen]
    snap_f = snap.loc[chosen]

    # 7) save families map
    fam_path = os.path.join(OUT_DIR, "variant_families.csv")
    families_df.to_csv(fam_path, index=False)
    print(f"🧬 variant families -> {fam_path}  (families={families_df['Family'].nunique()}, kept={len(chosen)}, dropped={len(tickers)-len(chosen)})")
    if drop_wu:
        print(f"   dropped warrants/units: {len(drop_wu)}")
    return mat_f, snap_f, families_df

# ==== core loaders & flip-corr ====
def load_matrix_from_db(span=SPAN, lookback_days=LOOKBACK_DAYS):
    if USE_VECTORIZED:
        with sqlite3.connect(VEC_DB_PATH) as conn:
            df = pd.read_sql(
                """SELECT begins_at, ticker, span,
                          pct_1d, trend_slope_60d, ret_60d, vol_60d, dollar_vol_20d
                   FROM VectorizedFeatures
                   WHERE span = ?""",
                conn, params=(span,), parse_dates=["begins_at"]
            )
        if df.empty:
            raise RuntimeError("VectorizedFeatures empty for span="+span)
        max_dt = df["begins_at"].max()
        cut_dt = max_dt - pd.Timedelta(days=lookback_days)
        df = df[df["begins_at"] >= cut_dt].copy()
        for c in ["pct_1d","trend_slope_60d","ret_60d","vol_60d","dollar_vol_20d"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["dir"] = np.sign(df["pct_1d"].values.astype(float))
        mat = df.pivot(index="begins_at", columns="ticker", values="dir").sort_index()
        snap = (df.sort_values("begins_at")
                  .groupby("ticker")
                  .tail(1)[["ticker","trend_slope_60d","ret_60d","vol_60d","dollar_vol_20d"]]
                  .set_index("ticker"))
        return mat, snap
    else:
        with sqlite3.connect(HIST_DB_PATH) as conn:
            hp = pd.read_sql(
                """SELECT begins_at, ticker, span, close_price
                   FROM HistoricalPrices WHERE span=?""",
                conn, params=(span,), parse_dates=["begins_at"]
            )
        if hp.empty:
            raise RuntimeError("HistoricalPrices empty for span="+span)
        max_dt = hp["begins_at"].max()
        cut_dt = max_dt - pd.Timedelta(days=lookback_days)
        hp = hp[hp["begins_at"] >= cut_dt].copy()
        hp["close_price"] = pd.to_numeric(hp["close_price"], errors="coerce")
        hp = hp.sort_values(["ticker","begins_at"])
        hp["pct_1d"] = hp.groupby("ticker")["close_price"].pct_change()
        hp["dir"] = np.sign(hp["pct_1d"])
        mat = hp.pivot(index="begins_at", columns="ticker", values="dir").sort_index()
        snap = (hp.sort_values("begins_at")
                  .groupby("ticker")
                  .tail(1)[["ticker"]]
                  .assign(trend_slope_60d=np.nan, ret_60d=np.nan, vol_60d=np.nan,
                          dollar_vol_20d=np.nan)
                  .set_index("ticker"))
        return mat, snap

def pairwise_corr_on_signs(mat, min_overlap=MIN_OVERLAP):
    tickers = mat.columns.tolist()
    X = mat.to_numpy(dtype=float)
    N = len(tickers)
    corr = np.full((N, N), np.nan, dtype=float)
    for i in tqdm(range(N), desc="pairwise corr rows"):
        xi = X[:, i]
        for j in range(i, N):
            xj = X[:, j]
            mask = ~(np.isnan(xi) | np.isnan(xj))
            n = int(mask.sum())
            if n < min_overlap:
                continue
            vi = xi[mask]; vj = xj[mask]
            if np.nanstd(vi) == 0 or np.nanstd(vj) == 0:
                continue
            c = np.corrcoef(vi, vj)[0,1]
            corr[i, j] = c
            corr[j, i] = c
    return tickers, corr

def build_pairs(tickers, corr, snap, sim_min=SIM_MIN, sim_cap=SIM_CAP, top_k=TOP_N_PER_TICKER):
    N = len(tickers)
    rows = []
    for i in tqdm(range(N), desc="build candidate pairs"):
        vals = []
        for j in range(N):
            if j == i: continue
            c = corr[i, j]
            if np.isnan(c): continue
            if c < sim_min or c >= sim_cap:
                continue
            vals.append((j, c))
        vals.sort(key=lambda t: t[1], reverse=True)
        vals = vals[:top_k]
        ai = tickers[i]
        for j, c in vals:
            aj = tickers[j]
            A, B = sorted([ai, aj])
            rows.append((A, B, c))

    if not rows:
        return pd.DataFrame(columns=["A","B","similarity","A_slope","B_slope","A_ret60","B_ret60",
                                     "A_vol60","B_vol60","A_dv20","B_dv20","A_sector","B_sector","winner"])

    pairs = pd.DataFrame(rows, columns=["A","B","similarity"]).drop_duplicates(subset=["A","B"])
    for side in ["A","B"]:
        pairs[f"{side}_slope"]  = snap.reindex(pairs[side]).trend_slope_60d.values
        pairs[f"{side}_ret60"]  = snap.reindex(pairs[side]).ret_60d.values
        pairs[f"{side}_vol60"]  = snap.reindex(pairs[side]).vol_60d.values
        pairs[f"{side}_dv20"]   = snap.reindex(pairs[side]).dollar_vol_20d.values

    def choose(row):
        sA, sB = row["A_slope"], row["B_slope"]
        if pd.notna(sA) and pd.notna(sB) and sA != sB:
            return row["A"] if sA > sB else row["B"]
        rA, rB = row["A_ret60"], row["B_ret60"]
        if pd.notna(rA) and pd.notna(rB) and rA != rB:
            return row["A"] if rA > rB else row["B"]
        dA, dB = row["A_dv20"], row["B_dv20"]
        if pd.notna(dA) and pd.notna(dB) and dA != dB:
            return row["A"] if dA > dB else row["B"]
        vA, vB = row["A_vol60"], row["B_vol60"]
        if pd.notna(vA) and pd.notna(vB) and vA != vB:
            return row["A"] if vA < vB else row["B"]
        return row["A"]

    pairs["winner"] = pairs.apply(choose, axis=1)
    return pairs.sort_values("similarity", ascending=False).reset_index(drop=True)

def winners_dedup(pairs):
    if pairs.empty:
        return pd.DataFrame(columns=["Ticker"])
    return pairs[["winner"]].drop_duplicates().rename(columns={"winner":"Ticker"}).reset_index(drop=True)

# ==== run ====
mat, snap = load_matrix_from_db()
print(f"Universe loaded: {mat.shape[1]} tickers, {mat.shape[0]} dates")

if ENABLE_VARIANT_DEDUP:
    mat, snap, fam_df = _variant_dedup(mat, snap)
    print(f"After variant dedup: {mat.shape[1]} tickers")

tickers, corr = pairwise_corr_on_signs(mat, MIN_OVERLAP)
pairs = build_pairs(tickers, corr, snap, SIM_MIN, SIM_CAP, TOP_N_PER_TICKER)
winners = winners_dedup(pairs)

pairs_out   = os.path.join(OUT_DIR, "flipcorr_pairs_5y.csv")
winners_out = os.path.join(OUT_DIR, "flipcorr_winners_5y.csv")
pairs.to_csv(pairs_out, index=False)
winners.to_csv(winners_out, index=False)
print(f"💾 saved pairs -> {pairs_out}  ({len(pairs)} rows)")
print(f"💾 saved winners -> {winners_out}  ({len(winners)} tickers)")

# optional heatmap
if TRY_PLOTLY:
    try:
        import plotly.graph_objects as go
        corr_df = pd.DataFrame(corr, index=tickers, columns=tickers)
        fig = go.Figure(data=go.Heatmap(
            z=corr_df.values,
            x=corr_df.columns,
            y=corr_df.index,
            zmin=-1, zmax=1, colorscale="RdBu",
            colorbar_title="flip corr"
        ))
        fig.update_layout(
            title=f"Flip-based similarity (span={SPAN}, lookback={LOOKBACK_DAYS}d)",
            xaxis_nticks=20, yaxis_nticks=20, height=900, width=1000
        )
        html_path = os.path.join(OUT_DIR, "flipcorr_5y_heatmap.html")
        fig.write_html(html_path, include_plotlyjs="cdn")
        print("📈 heatmap ->", html_path)
    except Exception as e:
        print("⚠️ Heatmap skipped:", e)


In [ ]:
# === Enrich winners with DB metrics, then shortlist (robust/chunked) ===
import os, sqlite3
import numpy as np, pandas as pd
from tqdm import tqdm

# -------------------- Paths --------------------
BASE_DIR       = str(globals().get("DATA_ROOT", PROJECT_ROOT))
OUT_DIR        = os.path.join(BASE_DIR, "analytics"); os.makedirs(OUT_DIR, exist_ok=True)
WINNERS_CSV    = os.path.join(OUT_DIR, "flipcorr_winners_5y.csv")
FAMILIES_CSV   = os.path.join(OUT_DIR, "variant_families.csv")   # <— NEW: respect variant dedup if present

HIST_DB_PATH   = os.path.join(BASE_DIR, "historicals.db")
VEC_DB_PATH    = os.path.join(BASE_DIR, "vectorized.db")
# -------------------- Config knobs --------------------
SPAN                 = "5year"
LOOKBACK_LIQ_DAYS    = 60          # liquidity lookback window
MIN_AVG_VOLUME       = 200_000     # shares/day
MIN_AVG_DOLLAR_VOL   = 2_000_000   # $/day
REQUIRE_POS_SLOPE    = True        # require positive 60d slope
MAX_VOL_60D          = 0.08        # cap on 60d daily vol (std of returns); None to disable
SHORTLIST_LIMIT      = 5           # keep the final daily list focused
DT_FMT               = "%Y-%m-%d %H:%M:%S"
CHUNK_SIZE           = 800         # stay under SQLite ~999 variable limit


REQUESTED_SPAN = SPAN

def choose_populated_span(db_path, table, requested_span):
    try:
        with sqlite3.connect(db_path) as conn:
            counts = pd.read_sql(
                f"""SELECT span, COUNT(*) AS rows, MAX(begins_at) AS max_dt
                    FROM {table}
                    GROUP BY span""",
                conn,
            )
    except Exception as exc:
        raise RuntimeError(f"Could not inspect {db_path}:{table}: {exc}")
    if counts.empty or not (counts["rows"] > 0).any():
        raise RuntimeError(f"{table} has no populated spans in {db_path}")
    if ((counts["span"] == requested_span) & (counts["rows"] > 0)).any():
        return requested_span
    pref = {"5year": 5, "year": 4, "3month": 3, "month": 2, "week": 1}
    ranked = counts[counts["rows"] > 0].copy()
    ranked["pref"] = ranked["span"].map(pref).fillna(0)
    ranked = ranked.sort_values(["max_dt", "pref", "rows"], ascending=[False, False, False])
    chosen = ranked.iloc[0]["span"]
    print(f"Requested span={requested_span!r} unavailable for enrichment; using span={chosen!r}.")
    return chosen

SPAN = choose_populated_span(VEC_DB_PATH, "VectorizedFeatures", REQUESTED_SPAN)

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

# -------------------- Load winners --------------------
winners = pd.read_csv(WINNERS_CSV)
if winners.empty or "Ticker" not in winners.columns:
    raise RuntimeError("Winners CSV not found or empty. Expected column 'Ticker'.")

# NEW: Respect variant dedup if the audit file exists
if os.path.exists(FAMILIES_CSV):
    fam = pd.read_csv(FAMILIES_CSV)
    chosen = set(fam.loc[fam["Chosen"]==True, "Member"].str.upper())
    winners = winners[winners["Ticker"].str.upper().isin(chosen)].reset_index(drop=True)

tickers = winners["Ticker"].dropna().unique().tolist()
print(f"🎯 winners loaded after variant check: {len(tickers)}")

# -------------------- Latest snapshot from VectorizedFeatures (chunked, no join) --------------------
snap_frames = []
with sqlite3.connect(VEC_DB_PATH) as conn:
    for chunk in tqdm(list(chunked(tickers, CHUNK_SIZE)), desc="loading latest features"):
        ph  = ",".join(["?"]*len(chunk))
        sql = f"""
            SELECT ticker, begins_at,
                   trend_slope_60d, vol_60d, dollar_vol_20d, ret_60d
            FROM VectorizedFeatures
            WHERE span=? AND ticker IN ({ph})
            ORDER BY ticker, begins_at
        """
        df = pd.read_sql(sql, conn, params=[SPAN]+chunk, parse_dates=["begins_at"])
        if not df.empty:
            df = (df.dropna(subset=["trend_slope_60d"])
                    .sort_values(["ticker","begins_at"])
                    .groupby("ticker").tail(1))
            snap_frames.append(df)

latest_rows = (pd.concat(snap_frames, ignore_index=True)
               if snap_frames else
               pd.DataFrame(columns=["ticker","begins_at","trend_slope_60d","vol_60d","dollar_vol_20d","ret_60d"]))

# -------------------- 60d liquidity from HistoricalPrices (chunked) --------------------
with sqlite3.connect(HIST_DB_PATH) as conn:
    last_dt = pd.read_sql(
        "SELECT MAX(begins_at) AS m FROM HistoricalPrices WHERE span=?",
        conn, params=(SPAN,), parse_dates=["m"]
    ).iloc[0,0]
    if pd.isna(last_dt):
        raise RuntimeError(f"No data in HistoricalPrices for span='{SPAN}'")
    cut_dt_str = (last_dt - pd.Timedelta(days=LOOKBACK_LIQ_DAYS)).strftime(DT_FMT)

liq_frames = []
with sqlite3.connect(HIST_DB_PATH) as conn:
    for chunk in tqdm(list(chunked(tickers, CHUNK_SIZE)), desc="pulling 60d liquidity"):
        ph  = ",".join(["?"]*len(chunk))
        sql = f"""
            SELECT begins_at, ticker, close_price, volume
            FROM HistoricalPrices
            WHERE span=? AND begins_at >= ? AND ticker IN ({ph})
        """
        part = pd.read_sql(sql, conn, params=[SPAN, cut_dt_str]+chunk, parse_dates=["begins_at"])
        liq_frames.append(part)

hp = (pd.concat(liq_frames, ignore_index=True)
      if liq_frames else pd.DataFrame(columns=["begins_at","ticker","close_price","volume"]))
hp["close_price"] = pd.to_numeric(hp["close_price"], errors="coerce")
hp["volume"]      = pd.to_numeric(hp["volume"], errors="coerce")
hp["dollar"]      = hp["close_price"] * hp["volume"]

liq = (hp.groupby("ticker")
         .agg(AvgVolume=("volume","mean"),
              AvgDollarVol=("dollar","mean"),
              Days=("volume","count"))
         .reset_index())

# -------------------- Merge everything --------------------
enriched = (winners
            .merge(latest_rows, left_on="Ticker", right_on="ticker", how="left")
            .drop(columns=["ticker"])
            .merge(liq, left_on="Ticker", right_on="ticker", how="left")
            .drop(columns=["ticker"])
           )

# numeric coercion
for c in ["trend_slope_60d","vol_60d","dollar_vol_20d","ret_60d","AvgVolume","AvgDollarVol"]:
    if c in enriched.columns:
        enriched[c] = pd.to_numeric(enriched[c], errors="coerce")

# -------------------- Build shortlist --------------------
short = enriched.copy()

# Strong sanity filter: ensure we actually had data in the 60d window
if "Days" in short.columns:
    short = short[short["Days"] >= max(20, LOOKBACK_LIQ_DAYS*0.6)]

# liquidity rules
short = short[(short["AvgVolume"] >= MIN_AVG_VOLUME) & (short["AvgDollarVol"] >= MIN_AVG_DOLLAR_VOL)]

# momentum/risk rules
if REQUIRE_POS_SLOPE and "trend_slope_60d" in short.columns:
    short = short[short["trend_slope_60d"] > 0]
if (MAX_VOL_60D is not None) and ("vol_60d" in short.columns):
    short = short[short["vol_60d"] <= MAX_VOL_60D]

# rank by slope -> ret60 -> dollar vol
rank_cols = [c for c in ["trend_slope_60d","ret_60d","AvgDollarVol"] if c in short.columns]
if rank_cols:
    short = short.sort_values(by=rank_cols, ascending=[False]*len(rank_cols))

short = short.head(SHORTLIST_LIMIT)

# -------------------- Save outputs --------------------
enriched_csv = os.path.join(OUT_DIR, "winners_enriched.csv")
short_csv    = os.path.join(OUT_DIR, "winners_shortlist.csv")
enriched.to_csv(enriched_csv, index=False)
short.to_csv(short_csv, index=False)
print(f"💾 enriched -> {enriched_csv} ({len(enriched)} rows)")
print(f"💾 shortlist -> {short_csv} ({len(short)} rows)")

# also store shortlist into vectorized.db for next steps
with sqlite3.connect(VEC_DB_PATH) as conn:
    tbl = short.rename(columns={"Ticker":"ticker"}).copy()
    tbl.to_sql("WinnerUniverse", conn, if_exists="replace", index=False)
    try:
        conn.execute("CREATE INDEX IF NOT EXISTS idx_winner_universe_ticker ON WinnerUniverse(ticker)")
    except Exception:
        pass
print("🗂️ vectorized.db → WinnerUniverse refreshed")


In [ ]:
import os, sqlite3, pandas as pd
import matplotlib.pyplot as plt

# --- Paths ---
BASE_DIR = str(PROJECT_ROOT)
SHORTLIST_PATH = os.path.join(BASE_DIR, "analytics", "winners_shortlist.csv")
HIST_DB_PATH   = os.path.join(BASE_DIR, "historicals.db")

SPAN = "5year"  # change to 'year'/'3month' if you want shorter windows

# --- Load shortlist tickers ---
sl = pd.read_csv(SHORTLIST_PATH)
if sl.empty or "Ticker" not in sl.columns:
    raise RuntimeError("Shortlist CSV not found or empty. Expected column 'Ticker'.")
tickers = sl["Ticker"].dropna().tolist()
print("Shortlist:", tickers)

# --- Helper: load price history from HistoricalPrices ---
def load_prices(ticker, span=SPAN):
    with sqlite3.connect(HIST_DB_PATH) as conn:
        df = pd.read_sql(
            """
            SELECT begins_at, close_price
            FROM HistoricalPrices
            WHERE ticker = ? AND span = ?
            ORDER BY begins_at
            """,
            conn, params=(ticker, span), parse_dates=["begins_at"]
        )
    if df.empty:
        return df
    df["close_price"] = pd.to_numeric(df["close_price"], errors="coerce")
    df = df.dropna(subset=["close_price"])
    return df

# --- 1) One chart per ticker, with 50-day MA ---
for t in tickers:
    df = load_prices(t)
    if df.empty:
        print(f"⚠️ no data for {t} ({SPAN})")
        continue

    df["MA50"] = df["close_price"].rolling(50).mean()

    plt.figure(figsize=(10, 5))
    plt.plot(df["begins_at"], df["close_price"], label=f"{t} close", linewidth=1.8)
    plt.plot(df["begins_at"], df["MA50"], label="MA(50)", linewidth=1.2, alpha=0.8)
    plt.title(f"{t} — {SPAN} trend")
    plt.xlabel("Date"); plt.ylabel("Price ($)")
    plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout()
    plt.show()

# --- 2) Overlay chart: all shortlist tickers rebased to 100 ---
series = []
for t in tickers:
    df = load_prices(t)
    if df.empty: 
        continue
    df = df.set_index("begins_at")[["close_price"]].rename(columns={"close_price": t})
    series.append(df)

if series:
    aligned = pd.concat(series, axis=1).dropna(how="any")
    if not aligned.empty:
        rebased = aligned / aligned.iloc[0] * 100.0  # start = 100

        plt.figure(figsize=(11, 6))
        for col in rebased.columns:
            plt.plot(rebased.index, rebased[col], label=col, linewidth=1.5)
        plt.title(f"Shortlist overlay (rebased to 100) — {SPAN}")
        plt.xlabel("Date"); plt.ylabel("Index (start=100)")
        plt.grid(True, alpha=0.3); plt.legend(ncol=2, fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ No overlapping dates across shortlist to plot overlay.")
else:
    print("⚠️ No overlapping data to plot overlay.")


In [ ]:
# === Enrich winners with DB metrics, then shortlist (safe merges) ===
import os, sqlite3
import numpy as np, pandas as pd
from tqdm import tqdm

# -------------------- Paths --------------------
BASE_DIR       = str(globals().get("DATA_ROOT", PROJECT_ROOT))
OUT_DIR        = os.path.join(BASE_DIR, "analytics"); os.makedirs(OUT_DIR, exist_ok=True)
WINNERS_CSVS   = [  # we'll use the first one that exists
    os.path.join(OUT_DIR, "flipcorr_winners_5y.csv"),
    os.path.join(OUT_DIR, "winners_shortlist.csv"),
]
FAMILIES_CSV   = os.path.join(OUT_DIR, "variant_families.csv")

HIST_DB_PATH   = os.path.join(BASE_DIR, "historicals.db")
VEC_DB_PATH    = os.path.join(BASE_DIR, "vectorized.db")
# -------------------- Knobs --------------------
SPAN               = "5year"
LOOKBACK_LIQ_DAYS  = 60
MIN_AVG_VOLUME     = 200_000
MIN_AVG_DOLLAR_VOL = 2_000_000
REQUIRE_POS_SLOPE  = True
MAX_VOL_60D        = 0.08
SHORTLIST_LIMIT    = 5
DT_FMT             = "%Y-%m-%d %H:%M:%S"
CHUNK_SIZE         = 800


REQUESTED_SPAN = SPAN

def choose_populated_span(db_path, table, requested_span):
    try:
        with sqlite3.connect(db_path) as conn:
            counts = pd.read_sql(
                f"""SELECT span, COUNT(*) AS rows, MAX(begins_at) AS max_dt
                    FROM {table}
                    GROUP BY span""",
                conn,
            )
    except Exception as exc:
        raise RuntimeError(f"Could not inspect {db_path}:{table}: {exc}")
    if counts.empty or not (counts["rows"] > 0).any():
        raise RuntimeError(f"{table} has no populated spans in {db_path}")
    if ((counts["span"] == requested_span) & (counts["rows"] > 0)).any():
        return requested_span
    pref = {"5year": 5, "year": 4, "3month": 3, "month": 2, "week": 1}
    ranked = counts[counts["rows"] > 0].copy()
    ranked["pref"] = ranked["span"].map(pref).fillna(0)
    ranked = ranked.sort_values(["max_dt", "pref", "rows"], ascending=[False, False, False])
    chosen = ranked.iloc[0]["span"]
    print(f"Requested span={requested_span!r} unavailable for enrichment; using span={chosen!r}.")
    return chosen

SPAN = choose_populated_span(VEC_DB_PATH, "VectorizedFeatures", REQUESTED_SPAN)

# -------------------- Helpers --------------------
def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def _norm_ticker_cols(df):
    """Return a copy with a single 'ticker' col (uppercased), dropping any other Ticker variants."""
    d = df.copy()
    matches = [c for c in d.columns if c.lower() in ("ticker","symbol")]
    if not matches:
        return d
    first = matches[0]
    d = d.rename(columns={first: "ticker"})
    drop_rest = [c for c in matches if c != first]
    if drop_rest:
        d = d.drop(columns=drop_rest)
    d["ticker"] = d["ticker"].astype(str).str.upper()
    d = d.loc[:, ~d.columns.duplicated()]  # remove any dup header weirdness from CSVs
    return d

def _safe_merge(left, right, key="ticker", suffix="_r"):
    """Merge on key; if non-key columns overlap, suffix the RIGHT columns to avoid ValueError."""
    L = _norm_ticker_cols(left)
    R = _norm_ticker_cols(right)
    overlap = (set(L.columns) & set(R.columns)) - {key}
    if overlap:
        R = R.rename(columns={c: f"{c}{suffix}" for c in overlap})
    return L.merge(R, on=key, how="left")

# -------------------- Load winners (normalize to 'ticker') --------------------
winners = None
for p in WINNERS_CSVS:
    if os.path.exists(p):
        winners = pd.read_csv(p)
        break
if winners is None or winners.empty:
    raise RuntimeError("Couldn't find winners CSV. Expected analytics/flipcorr_winners_5y.csv or winners_shortlist.csv")

winners = _norm_ticker_cols(winners)
if "ticker" not in winners.columns:
    raise RuntimeError("Winners CSV must contain a 'Ticker' or 'ticker' or 'symbol' column.")
winners = winners[["ticker"]].dropna().drop_duplicates()

# respect variant families if present (keep only chosen members)
if os.path.exists(FAMILIES_CSV):
    fam = pd.read_csv(FAMILIES_CSV)
    chosen = set(fam.loc[fam["Chosen"]==True, "Member"].astype(str).str.upper())
    winners = winners[winners["ticker"].isin(chosen)].reset_index(drop=True)

tickers = winners["ticker"].tolist()
print(f"🎯 winners loaded after variant check: {len(tickers)}")

# -------------------- Latest snapshot from VectorizedFeatures --------------------
snap_frames = []
with sqlite3.connect(VEC_DB_PATH) as conn:
    for chunk in tqdm(list(chunked(tickers, CHUNK_SIZE)), desc="loading latest features"):
        ph  = ",".join(["?"]*len(chunk))
        sql = f"""
            SELECT UPPER(ticker) AS ticker, begins_at,
                   trend_slope_60d, vol_60d, dollar_vol_20d, ret_60d
            FROM VectorizedFeatures
            WHERE span=? AND UPPER(ticker) IN ({ph})
            ORDER BY ticker, begins_at
        """
        df = pd.read_sql(sql, conn, params=[SPAN]+chunk, parse_dates=["begins_at"])
        if not df.empty:
            df = (df.dropna(subset=["trend_slope_60d"])
                    .sort_values(["ticker","begins_at"])
                    .groupby("ticker").tail(1))
            snap_frames.append(df)

latest_rows = (pd.concat(snap_frames, ignore_index=True)
               if snap_frames else
               pd.DataFrame(columns=["ticker","begins_at","trend_slope_60d","vol_60d","dollar_vol_20d","ret_60d"]))

# -------------------- 60d liquidity from HistoricalPrices --------------------
with sqlite3.connect(HIST_DB_PATH) as conn:
    last_dt = pd.read_sql(
        "SELECT MAX(begins_at) AS m FROM HistoricalPrices WHERE span=?",
        conn, params=(SPAN,), parse_dates=["m"]
    ).iloc[0,0]
    if pd.isna(last_dt):
        raise RuntimeError(f"No data in HistoricalPrices for span='{SPAN}'")
    cut_dt_str = (last_dt - pd.Timedelta(days=LOOKBACK_LIQ_DAYS)).strftime(DT_FMT)

liq_frames = []
with sqlite3.connect(HIST_DB_PATH) as conn:
    for chunk in tqdm(list(chunked(tickers, CHUNK_SIZE)), desc="pulling 60d liquidity"):
        ph  = ",".join(["?"]*len(chunk))
        sql = f"""
            SELECT begins_at, UPPER(ticker) AS ticker, close_price, volume
            FROM HistoricalPrices
            WHERE span=? AND begins_at >= ? AND UPPER(ticker) IN ({ph})
        """
        part = pd.read_sql(sql, conn, params=[SPAN, cut_dt_str]+chunk, parse_dates=["begins_at"])
        liq_frames.append(part)

hp = (pd.concat(liq_frames, ignore_index=True)
      if liq_frames else pd.DataFrame(columns=["begins_at","ticker","close_price","volume"]))
hp["close_price"] = pd.to_numeric(hp["close_price"], errors="coerce")
hp["volume"]      = pd.to_numeric(hp["volume"], errors="coerce")
hp["dollar"]      = hp["close_price"] * hp["volume"]

liq = (hp.groupby("ticker")
         .agg(AvgVolume=("volume","mean"),
              AvgDollarVol=("dollar","mean"),
              Days=("volume","count"))
         .reset_index())

# -------------------- SAFE MERGES (no overlap errors possible) --------------------
enriched = winners.copy()
enriched = _safe_merge(enriched, latest_rows, suffix="_v")
enriched = _safe_merge(enriched, liq,         suffix="_liq")

# numeric coercion
for c in ["trend_slope_60d","vol_60d","dollar_vol_20d","ret_60d","AvgVolume","AvgDollarVol"]:
    if c in enriched.columns:
        enriched[c] = pd.to_numeric(enriched[c], errors="coerce")

# -------------------- Build shortlist --------------------
short = enriched.copy()

# ensure we saw enough recent days for liquidity calc
if "Days" in short.columns:
    short = short[short["Days"] >= max(20, int(LOOKBACK_LIQ_DAYS*0.6))]

# liquidity rules
short = short[(short["AvgVolume"] >= MIN_AVG_VOLUME) & (short["AvgDollarVol"] >= MIN_AVG_DOLLAR_VOL)]

# momentum/risk rules
if REQUIRE_POS_SLOPE and "trend_slope_60d" in short.columns:
    short = short[short["trend_slope_60d"] > 0]
if (MAX_VOL_60D is not None) and ("vol_60d" in short.columns):
    short = short[short["vol_60d"] <= MAX_VOL_60D]

# rank by slope -> ret60 -> dollar vol
rank_cols = [c for c in ["trend_slope_60d","ret_60d","AvgDollarVol"] if c in short.columns]
if rank_cols:
    short = short.sort_values(by=rank_cols, ascending=[False]*len(rank_cols))

short = short.head(SHORTLIST_LIMIT)

# -------------------- Save outputs --------------------
enriched_out = enriched.rename(columns={"ticker":"Ticker"})
short_out    = short.rename(columns={"ticker":"Ticker"})

enriched_csv = os.path.join(OUT_DIR, "winners_enriched.csv")
short_csv    = os.path.join(OUT_DIR, "winners_shortlist.csv")
enriched_out.to_csv(enriched_csv, index=False)
short_out.to_csv(short_csv, index=False)
print(f"💾 enriched -> {enriched_csv} ({len(enriched_out)} rows)")
print(f"💾 shortlist -> {short_csv} ({len(short_out)} rows)")

# also store shortlist into vectorized.db for next steps
with sqlite3.connect(VEC_DB_PATH) as conn:
    tbl = short_out.rename(columns={"Ticker":"ticker"}).copy()
    tbl.to_sql("WinnerUniverse", conn, if_exists="replace", index=False)
    try:
        conn.execute("CREATE INDEX IF NOT EXISTS idx_winner_universe_ticker ON WinnerUniverse(ticker)")
    except Exception:
        pass
print("🗂️ vectorized.db → WinnerUniverse refreshed")


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = PROJECT_ROOT / "vectorized.db"

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# 1) list tables
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

print("TABLES:")
display(tables)

for table in tables["name"]:
    print(f"\n{'='*80}")
    print(f"TABLE: {table}")

    # 2) schema
    schema = pd.read_sql(f"PRAGMA table_info({table})", conn)
    print("\nCOLUMNS:")
    display(schema)

    # 3) row count
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table}", conn)
    print("\nROW COUNT:")
    display(count)

    # 4) sample rows
    sample = pd.read_sql(f"SELECT * FROM {table} LIMIT 5", conn)
    print("\nSAMPLE ROWS:")
    display(sample)

conn.close()

In [ ]:
db_path = PROJECT_ROOT / "vectorized.db"

In [ ]:
db_path = PROJECT_ROOT / "filtered_tickers.db"

In [ ]:
db_path = PROJECT_ROOT / "vectorized.db"